In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Load data
train_aug = ImageDataGenerator(
    rescale=1./255, rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, zoom_range=0.1,
    validation_split=0.15)
val_aug = ImageDataGenerator(rescale=1./255, validation_split=0.15)

common = dict(target_size=(48, 48), color_mode='grayscale',
              class_mode='categorical', batch_size=64, seed=42)

train_data = train_aug.flow_from_directory('C:/Users/mahig/OneDrive/Desktop/Nexora/emotion-model/data/fer2013/train', subset='training', **common)
val_data = val_aug.flow_from_directory('C:/Users/mahig/OneDrive/Desktop/Nexora/emotion-model/data/fer2013/train', subset='validation',
                                       shuffle=False, **common)
test_data = ImageDataGenerator(rescale=1./255).flow_from_directory(
    'C:/Users/mahig/OneDrive/Desktop/Nexora/emotion-model/data/fer2013/test', shuffle=False, **common)

print(train_data.class_indices)   # save this label order!

# 2. Define the model
model = models.Sequential([
    layers.Input((48, 48, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

# 3. Compile and train (your code)
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5),
    tf.keras.callbacks.ModelCheckpoint('best.keras', save_best_only=True)
]

model.fit(train_data, validation_data=val_data, epochs=50, callbacks=callbacks)

# 4. Final check on unseen data
print(model.evaluate(test_data))

Found 24406 images belonging to 7 classes.
Found 4303 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.
{'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
Epoch 1/50
382/382 ━━━━━━━━━━━━━━━━━━━━ 112s 282ms/step - accuracy: 0.2449 - loss: 1.9157 - val_accuracy: 0.2336 - val_loss: 1.8237 - learning_rate: 0.0010
Epoch 2/50
382/382 ━━━━━━━━━━━━━━━━━━━━ 132s 346ms/step - accuracy: 0.2721 - loss: 1.7626 - val_accuracy: 0.3126 - val_loss: 1.6629 - learning_rate: 0.0010
Epoch 3/50
382/382 ━━━━━━━━━━━━━━━━━━━━ 98s 257ms/step - accuracy: 0.2952 - loss: 1.7064 - val_accuracy: 0.3804 - val_loss: 1.5920 - learning_rate: 0.0010
Epoch 4/50
382/382 ━━━━━━━━━━━━━━━━━━━━ 136s 242ms/step - accuracy: 0.3150 - loss: 1.6643 - val_accuracy: 0.3353 - val_loss: 1.6800 - learning_rate: 0.0010
Epoch 5/50
382/382 ━━━━━━━━━━━━━━━━━━━━ 82s 214ms/step - accuracy: 0.3286 - loss: 1.6360 - val_accuracy: 0.3630 - val_loss: 1.5460 - learning_rate: 0.0010
Epoch 